Data Analyst: Andrei Berner Arroyo Tanta

**Project 4 with Numpy: Massive Portfolio Optimizer & Value at Risk (VaR) Engine

**Business Application: A diversified portfolio requires processing vast amounts of historical data while maintaining the correlation between assets. This project automates the cleaning and projection of a 50-asset portfolio.

-Business Need: Cleaning five years of historical data and projecting future value using Monte Carlo simulations.

-Impacted Area: Asset Management, Portfolio Strategy, and Financial Compliance.

-Decision Support: Allows managers to visualize potential loss scenarios and optimize asset weighting to stay within risk tolerances.

**Executive Summary: 

-The system processed a 3D data structure for 50 assets over 1,260 days, implementing a "Stability Corridor" to maintain data integrity.

-Key Findings: Automated vectorized cleaning identified and corrected 13,695 outliers without using slow iterative loops.Relevant Metrics:
    
    -Average Final Value: 1,917.71 dollars (a projected growth of +91.77%).
    
    -Value at Risk (VaR 95%): 971.10 dollars (indicating a maximum potential loss of -2.89% with 95% confidence).
    
    -Upside Potential (95%): 3,976.77 dollars.

-Detected Risks: The model identified high volatility (Standard Deviation of 2,307.73), suggesting that while growth is likely, the range of possible outcomes is broad, requiring active hedging strategies.

In [1]:
#Environment & Pre-allocation
import numpy as np

In [2]:
#Portfolio Configuration
n_assets = 50
n_days = 252 * 5 # 5-Year Horizon
initial_price = 317.59 # Benchmark: GOOG price (2026-02-01)
mean_return = 0.0005
volatility = 4.5

np.random.seed(42)

In [3]:
#Vectorized Path Generation
#Pre-allocating a (1259, 50) matrix for daily price fluctuations
shocks = np.random.normal(mean_return, volatility, size=(n_days - 1, n_assets))
price_paths_raw = np.cumsum(shocks, axis=0)

#Constructing the 2D Price Matrix
starting_row = np.full((1, n_assets), initial_price)
prices_stock = np.vstack([starting_row, initial_price + price_paths_raw])

print(f"Dataset Dimensions: {prices_stock.shape} (Days x Assets)")

Dataset Dimensions: (1260, 50) (Days x Assets)


In [4]:
#Vectorized Data Cleaning: The "Stability Corridor"

#Handle Missing Values (NaN)
#Replace NaNs with the median to maintain distributional integrity
prices_stock = np.where(np.isnan(prices_stock), np.median(prices_stock), prices_stock)

#Outlier Detection (Stability Corridor)
#Define technical boundaries (Support: 65%, Resistance: 140% of median)
median_val   = np.median(prices_stock)
bottom_limit = median_val * 0.65
top_limit    = median_val * 1.4

#Targeted Mitigation
outliers_mask = (prices_stock < bottom_limit) | (prices_stock > top_limit)
row_idx, col_idx = np.where(outliers_mask)

#Filter out day 0 to calculate percentage jumps from previous days
valid_move_idx = row_idx > 0
r, c = row_idx[valid_move_idx], col_idx[valid_move_idx]

#Calculate jumps (%) to differentiate between market trends and data errors
pct_change = np.abs((prices_stock[r, c] - prices_stock[r, c-1]) / prices_stock[r, c-1]) * 100
risk_limit = 30 # Threshold for unrealistic jumps

#Correction Logic:
#Jumps <= 30% are adjusted to the median (noise reduction)
#Jumps > 30% are capped at the technical top_limit
prices_stock[r, c] = np.where(pct_change <= risk_limit, median_val, prices_stock[r, c])
prices_stock[r, c] = np.where(pct_change > risk_limit, top_limit, prices_stock[r, c])

print(f"Total Outliers Corrected: {outliers_mask.sum()}")

Total Outliers Corrected: 13695


In [5]:
#Financial Metrics & Linear Algebra

#Calculate Daily Returns
price_diffs = np.diff(prices_stock, axis=0)
returns_matrix = price_diffs / prices_stock[:-1, :]

#Statistical Aggregates
avg_daily_returns = np.mean(returns_matrix, axis=0)
cov_matrix = np.cov(returns_matrix, rowvar=False)

#Lead Asset Performance (Fancy Indexing)
best_asset_idx = np.argmax(prices_stock[-1, :])
print(f"Top Performing Asset Index: {best_asset_idx}")
print(f"Asset Final Valuation: ${prices_stock[-1, best_asset_idx]:.2f}")

Top Performing Asset Index: 0
Asset Final Valuation: $443.06


In [6]:
#Scalable Monte Carlo Simulation (3D Array Logic)
n_simulations = 10000
trading_days  = 252 # 1-year projection

#Portfolio Weighting (Dirichlet Distribution ensures weights sum to 1.0)
weights = np.random.dirichlet(np.ones(n_assets), size=1)[0]

#Generate Simulated Returns using Multivariate Normal Distribution
#This preserves the historical correlation between assets (Crucial for Risk Analysis)
simulated_returns = np.random.multivariate_normal(
    mean = avg_daily_returns,
    cov  = cov_matrix,
    size = (n_simulations, trading_days)
)

#3D Vectorized Aggregation
#Calculate cumulative growth for every scenario across all assets
accumulated_growth = np.prod(1 + simulated_returns, axis=1)

#Project Portfolio Value (Initial Investment: $1000)
#We use np.dot to apply asset weights across all 10,000 simulations
portfolio_start_val = 1000
final_portfolio_values = portfolio_start_val * np.dot(accumulated_growth, weights)

In [7]:
#Risk Reporting & Statistical Summary
avg_projection = np.mean(final_portfolio_values)
var_95 = np.percentile(final_portfolio_values, 5)  # 5th percentile (95% confidence)
potential_95 = np.percentile(final_portfolio_values, 95) # 95th percentile

print(f"--- Monte Carlo Projections (1-Year) ---")
print(f"Average Final Value: ${avg_projection:,.2f} (+{((avg_projection-1000)/10):.2f}%)")
print(f"Value at Risk (VaR 95%): ${var_95:,.2f} (Max potential loss: {((var_95-1000)/10):.2f}%)")
print(f"Upside Potential (95%): ${potential_95:,.2f}")
print(f"Volatility (Standard Deviation): {np.std(final_portfolio_values):,.2f}")

--- Monte Carlo Projections (1-Year) ---
Average Final Value: $1,917.71 (+91.77%)
Value at Risk (VaR 95%): $971.10 (Max potential loss: -2.89%)
Upside Potential (95%): $3,976.77
Volatility (Standard Deviation): 2,307.73
